# 1. End-to-End Baseline Pipeline: Combined Candidates -> Content Similarity

Runs the combined candidate generator with the content-similarity reranker on validation
(and optionally test):

- **Candidate generation**: `CG_content_two_tower_rolling_popularity.ipynb` writes the
  Top-500 articles per request: 200 by rolling prior-hour click popularity, followed by
  300 content two-tower candidates.
- **Reranking**: `RR_content_similarity_baseline.ipynb` reorders those 500 candidates by
  cosine similarity to the user's causal clicked-history profile. Five rules are compared:
  text, entity, and fusion with text weight 0.25 / 0.50 / 0.75.

The final Top-K list is evaluated with all metrics in `cg_evaluation.py` at `M = K`,
plus MRR@K and nDCG@K from `rr_evaluation.py`.

**Labels are incomplete.** A retrieved article that was never shown has unknown relevance:
it counts as "not a known click", not as a negative. Every known click stays in the
denominator, so clicks the generator missed count as misses.

The best content rule is selected on validation by nDCG@K. Test reuses that selection.

## 1.1 Load Packages

In [1]:
import json
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display

## 1.2 Set Up Paths

`NEWSREC_ROOT` overrides the project checkout. `NEWSREC_DATA_ROOT` can point at a separate
data root; otherwise the checkout is used. `TARGET_SPLIT` selects validation or test. A test
run needs the validation selection artifact and a test candidate file from the rolling
popularity notebook.

In [2]:
CODE_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run_all.py").is_file()
)
ROOT = Path(os.environ.get("NEWSREC_ROOT", CODE_ROOT)).expanduser().resolve()
REQUESTED_DATA_ROOT = Path(os.environ.get("NEWSREC_DATA_ROOT", ROOT)).expanduser().resolve()
REPLAY_ROOT = (
    REQUESTED_DATA_ROOT
    if (REQUESTED_DATA_ROOT / "1_Data_Preparation" / "processed").exists()
    else ROOT
)

PROCESSED = REPLAY_ROOT / "1_Data_Preparation" / "processed"
REPRESENTATIONS = REPLAY_ROOT / "3_Representations" / "artifacts"
RUN = REPLAY_ROOT / "4_Feature_Engineering" / "artifacts" / "prequential-all"
CANDIDATE_ARTIFACTS = (
    ROOT
    / "5_Models"
    / "5_01_Candidate_Generation"
    / "artifacts"
    / "combined_content_two_tower_rolling_popularity"
)
OUTPUT_DIR = REPLAY_ROOT / "5_Models" / "5_03_Pipeline_Evaluation" / "artifacts"
RESULTS = REPLAY_ROOT / "results"
REPLAY_OUTPUT = OUTPUT_DIR / "replayed_features"

sys.path.insert(0, str(ROOT / "5_Models" / "5_01_Candidate_Generation"))
sys.path.insert(0, str(ROOT / "5_Models" / "5_02_Reranking"))
sys.path.insert(0, str(ROOT / "src"))

from cg_evaluation import CandidateEvaluator
from newsrec.common import DEFAULT_SETTINGS
from newsrec.pipeline import replay_batches, time_horizon
from newsrec.ranking import TemporalState, build_features, feature_schema
from newsrec.representations import Bundle
from rr_evaluation import evaluate_retrieved_candidates

for path in (OUTPUT_DIR, RESULTS, REPLAY_OUTPUT):
    path.mkdir(parents=True, exist_ok=True)

print("Repository:", ROOT)
print("Replay root:", REPLAY_ROOT)
print("Candidate folder:", CANDIDATE_ARTIFACTS)

Repository: C:\Users\Yong Rui\BT4222-Group-6-
Replay root: C:\Users\Yong Rui\BT4222-Group-6-
Candidate folder: C:\Users\Yong Rui\BT4222-Group-6-\5_Models\5_01_Candidate_Generation\artifacts\combined_content_two_tower_rolling_popularity


## 1.3 Configuration

In [3]:
TARGET_SPLIT = "validation"  # "validation" or "test"
K = 10
EVAL_IMPRESSIONS = 10_000  # None = every impression in the candidate file
SEED = 4222

CANDIDATE_CAP = 500
CANDIDATE_PATH = CANDIDATE_ARTIFACTS / f"recommendations_{TARGET_SPLIT}_top{CANDIDATE_CAP}.parquet"
SELECTION_PATH = (
    RESULTS
    / f"pipeline_{CANDIDATE_ARTIFACTS.name}_content_similarity_selection_validation.json"
)

CONTENT_COLUMNS = [
    "smoothed_prior_article_ctr",
    "text_cosine", "entity_cosine",
    "text_match_valid", "entity_match_valid",
    "no_text_profile", "no_entity_profile",
]
MISSING_SCORE = -2.0

TRIALS = {
    "content_text_cosine": {"kind": "text"},
    "content_entity_cosine": {"kind": "entity"},
    "content_fusion_text_0.25": {"kind": "fusion", "text_weight": 0.25},
    "content_fusion_text_0.50": {"kind": "fusion", "text_weight": 0.50},
    "content_fusion_text_0.75": {"kind": "fusion", "text_weight": 0.75},
}
SELECTION_METRIC = f"ndcg_at_{K}"
IMPRESSION_TAG = "all" if EVAL_IMPRESSIONS is None else f"{EVAL_IMPRESSIONS}imp"
BASE_MODEL_NAME = f"{CANDIDATE_ARTIFACTS.name}_content_similarity"

schema = json.loads((RUN / "feature_schema.json").read_text())
FEATURES = schema["model_columns"]
if feature_schema(DEFAULT_SETTINGS)["model_columns"] != FEATURES:
    raise ValueError("The feature code and the saved feature run disagree on the schema")
missing = set(CONTENT_COLUMNS) - set(FEATURES)
if missing:
    raise ValueError(f"Content columns missing from the feature schema: {sorted(missing)}")

print("Target split:", TARGET_SPLIT)
print("Final K:", K)
print("Evaluation impressions:", EVAL_IMPRESSIONS)
print("Selection metric:", SELECTION_METRIC)
print("Candidate file:", CANDIDATE_PATH)

Target split: validation
Final K: 10
Evaluation impressions: 10000
Selection metric: ndcg_at_10
Candidate file: C:\Users\Yong Rui\BT4222-Group-6-\5_Models\5_01_Candidate_Generation\artifacts\combined_content_two_tower_rolling_popularity\recommendations_validation_top500.parquet


## 1.4 Load Generated Candidates

In [4]:
if not CANDIDATE_PATH.exists():
    raise FileNotFoundError(
        f"Candidate file not found: {CANDIDATE_PATH}. Run "
        f"CG_content_two_tower_rolling_popularity.ipynb with TARGET_SPLIT='{TARGET_SPLIT}' first."
    )

candidates = (
    pl.read_parquet(CANDIDATE_PATH, columns=["impression_uid", "cutoff", "news_code"])
    .sort(["impression_uid", "cutoff"])
)
if candidates.select(["impression_uid", "news_code"]).n_unique() != candidates.height:
    raise ValueError("Candidate file contains duplicate impression/news pairs")

shape = candidates.group_by("impression_uid").agg(
    pl.len().alias("rows"),
    pl.col("cutoff").min().alias("first_cutoff"),
    pl.col("cutoff").max().alias("last_cutoff"),
    pl.col("cutoff").n_unique().alias("cutoffs"),
)
bad = shape.filter(
    (pl.col("rows") != CANDIDATE_CAP)
    | (pl.col("first_cutoff") != 1)
    | (pl.col("last_cutoff") != CANDIDATE_CAP)
    | (pl.col("cutoffs") != CANDIDATE_CAP)
)
if bad.height:
    raise ValueError(f"{bad.height} impressions have an invalid candidate shape")
if K > CANDIDATE_CAP:
    raise ValueError("K cannot exceed the candidate cap")

all_uids = candidates["impression_uid"].unique().sort().to_numpy()
if EVAL_IMPRESSIONS is not None:
    chosen = np.random.default_rng(SEED).choice(
        all_uids, size=min(EVAL_IMPRESSIONS, len(all_uids)), replace=False
    )
    candidates = candidates.filter(pl.col("impression_uid").is_in(np.sort(chosen).tolist()))

selected_uids = candidates["impression_uid"].unique().sort()
n_selected = selected_uids.len()
print(f"Impressions: {n_selected:,} of {len(all_uids):,}")
print(f"Candidate rows: {candidates.height:,}")
display(candidates.head(5))

Impressions: 10,000 of 31,624
Candidate rows: 5,000,000


impression_uid,cutoff,news_code
i64,i16,i32
10000019,1,287
10000019,2,51048
10000019,3,39010
10000019,4,59673
10000019,5,34876


# 2. Replay Causal Features

The combined generator's candidates are generated articles, not logged slate entries, so most
of them have no row in `ranking_features_*`. The causal timeline is replayed to build the
same content features used by `RR_content_similarity_baseline.ipynb`. Features for a request
use only history strictly before its timestamp, and candidates never update the state.

The cache holds only the seven columns the content rule needs, so re-runs reuse it.

In [5]:
def replay_content_features(split, candidate_frame):
    """Replay the causal timeline and cache the content features for generated candidates."""
    impression_count = candidate_frame["impression_uid"].n_unique()
    cache_path = (
        REPLAY_OUTPUT
        / f"{split}_{CANDIDATE_ARTIFACTS.name}_{impression_count}imp_"
        f"{candidate_frame.height}rows_content_features.parquet"
    )
    if cache_path.exists():
        print("Reusing replayed features:", cache_path.name)
        return cache_path

    frame = (
        candidate_frame.sort(["impression_uid", "cutoff"]).to_pandas().reset_index(drop=True)
    )
    rows_by_impression = frame.groupby("impression_uid", sort=False).indices
    target_ids = set(rows_by_impression)
    replay_splits = tuple(
        ("train", "validation", "test")[: ("train", "validation", "test").index(split) + 1]
    )

    bundle = Bundle(REPLAY_ROOT, DEFAULT_SETTINGS)
    state = TemporalState(bundle, *time_horizon(REPLAY_ROOT), DEFAULT_SETTINGS)
    partial_path = cache_path.with_suffix(".partial.parquet")
    writer, pending, impressions, started = None, [], 0, time.monotonic()
    output_columns = ["impression_uid", "news_code", "cutoff", *CONTENT_COLUMNS]

    def flush():
        nonlocal writer, pending
        if not pending:
            return
        out = pd.concat(pending, ignore_index=True)[output_columns]
        table = pa.Table.from_pandas(out, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(partial_path, table.schema, compression="zstd")
        writer.write_table(table)
        pending = []

    print(
        f"Replaying {split}: {len(frame):,} candidates for "
        f"{len(rows_by_impression):,} impressions"
    )
    try:
        for timestamp, requests, events in replay_batches(
            REPLAY_ROOT, DEFAULT_SETTINGS, splits=replay_splits
        ):
            state.begin(requests, timestamp)
            selected = requests[
                requests["split"].eq(split)
                & requests["impression_uid"].isin(target_ids)
            ]
            if len(selected):
                pairs = frame.iloc[
                    np.concatenate(
                        [rows_by_impression[uid] for uid in selected["impression_uid"]]
                    )
                ]
                features = build_features(
                    selected, pairs, state, bundle, DEFAULT_SETTINGS, "retrieved"
                ).drop(columns=["cutoff"], errors="ignore")
                pending.append(
                    features.merge(
                        pairs[["impression_uid", "news_code", "cutoff"]],
                        on=["impression_uid", "news_code"],
                        validate="one_to_one",
                    )
                )
                impressions += len(selected)
                if sum(len(chunk) for chunk in pending) >= 50_000:
                    flush()
                    print(
                        f"  {impressions:,}/{len(rows_by_impression):,} impressions "
                        f"({(time.monotonic() - started) / 60:.1f} min)",
                        flush=True,
                    )
            state.finish(requests, events, timestamp, set(replay_splits))
        flush()
    finally:
        if writer is not None:
            writer.close()
        state.close()

    if impressions != len(rows_by_impression):
        raise ValueError(
            f"Replayed {impressions:,} impressions but {len(rows_by_impression):,} were selected"
        )
    partial_path.replace(cache_path)
    print(f"Saved replayed features: {cache_path.name}")
    return cache_path


features_path = replay_content_features(TARGET_SPLIT, candidates)

Reusing replayed features: validation_combined_content_two_tower_rolling_popularity_10000imp_5000000rows_content_features.parquet


# 3. Content Similarity Reranking

The scoring rule is copied unchanged from `RR_content_similarity_baseline.ipynb`. A candidate
with no valid vector scores `-2`, below every valid cosine in `[-1, 1]`. If the user has no
profile at all, the rule falls back to the causal smoothed article CTR.

Each trial reorders all 500 candidates by score descending, with `news_code` ascending as the
deterministic tie-break, and keeps ranks `1..K`. The `generator_order` reference keeps the
combined generator ranks `1..K`.

In [6]:
def content_score(frame, trial):
    """Score candidates with one text, entity or fusion rule."""
    prior = frame["smoothed_prior_article_ctr"].to_numpy().astype(float)
    text = frame["text_cosine"].to_numpy().astype(float)
    entity = frame["entity_cosine"].to_numpy().astype(float)
    text_valid = frame["text_match_valid"].to_numpy().astype(bool)
    entity_valid = frame["entity_match_valid"].to_numpy().astype(bool)
    no_text_profile = frame["no_text_profile"].to_numpy().astype(bool)
    no_entity_profile = frame["no_entity_profile"].to_numpy().astype(bool)

    if trial["kind"] == "text":
        return np.where(no_text_profile, prior, np.where(text_valid, text, MISSING_SCORE))
    if trial["kind"] == "entity":
        return np.where(no_entity_profile, prior, np.where(entity_valid, entity, MISSING_SCORE))

    alpha = trial["text_weight"]
    blended = np.where(
        text_valid & entity_valid,
        alpha * text + (1 - alpha) * entity,
        np.where(text_valid, text, np.where(entity_valid, entity, MISSING_SCORE)),
    )
    return np.where(no_text_profile & no_entity_profile, prior, blended)


def rank_top_k(scored, trial):
    """Apply one content rule and keep each impression's final Top-K list."""
    scores = content_score(scored, trial)
    return (
        scored.with_columns(pl.Series("score", scores, dtype=pl.Float64))
        .sort(["impression_uid", "score", "news_code"], descending=[False, True, False])
        .with_columns(
            pl.int_range(1, pl.len() + 1).over("impression_uid").cast(pl.Int16).alias("cutoff")
        )
        .filter(pl.col("cutoff") <= K)
        .select("impression_uid", "cutoff", "news_code", "score")
    )


replayed = pl.read_parquet(
    features_path, columns=["impression_uid", "news_code", *CONTENT_COLUMNS]
)
scored_candidates = candidates.join(
    replayed, on=["impression_uid", "news_code"], how="left"
)
if scored_candidates.select(
    pl.any_horizontal(pl.col(CONTENT_COLUMNS).is_null()).sum()
).item():
    raise ValueError("Some generated candidates are missing replayed content features")

generator_top_k = (
    candidates.filter(pl.col("cutoff") <= K)
    .with_columns((-pl.col("cutoff").cast(pl.Float64)).alias("score"))
    .select("impression_uid", "cutoff", "news_code", "score")
)
print("Generator Top-K rows:", generator_top_k.height)

Generator Top-K rows: 100000


# 4. Evaluate the Pipeline

Three things are reported for every ordering of the same generated candidates:

- **All `cg_evaluation.py` metrics at `M = K`**: pooled and impression-mean recall, hit rate,
  long-tail recall, cold-article recall, novelty, catalogue coverage and personalisation.
- **MRR@K**: reciprocal rank of the first known click in the final Top-K list.
- **nDCG@K**: binary-click nDCG over the final Top-K list.

`generator_order` is the pipeline without the reranker. It is a reference row, not part of
the validation selection. The best content rule is chosen by validation nDCG@K and saved for
the test run.

In [7]:
evaluator = CandidateEvaluator(root=REPLAY_ROOT, split=TARGET_SPLIT)

CG_METRICS = [
    "recall_pooled",
    "recall_impression_mean",
    "hit_rate",
    "long_tail_recall",
    "cold_article_recall",
    "novelty_bits",
    "catalogue_coverage",
    "personalisation",
]


def evaluate_pipeline(top_k, model_name):
    """Combine the cg_evaluation metrics at K with MRR@K and nDCG@K."""
    cg = evaluator.evaluate(
        top_k.select("impression_uid", "cutoff", "news_code"),
        model_name=model_name,
        m_values=(K,),
        impression_uids=selected_uids,
    )
    cg_wide = (
        cg.pivot(on="metric", index=["model", "split", "m"], values="value")
        .rename({"m": "k"})
        .with_columns(pl.lit(n_selected).alias("impressions"))
    )
    rr = evaluate_retrieved_candidates(
        top_k.select("impression_uid", "news_code", "score"),
        feature_run=RUN,
        split=TARGET_SPLIT,
        score_column="score",
        model_name=model_name,
        k_values=(K,),
    ).select(
        "model",
        pl.col("impressions").alias("clicked_impressions"),
        "mrr",
        pl.col(f"ndcg_at_{K}"),
    )
    return cg_wide.join(rr, on="model", how="left")


if TARGET_SPLIT == "validation":
    trial_names = list(TRIALS)
else:
    if not SELECTION_PATH.exists():
        raise FileNotFoundError(
            f"Validation selection not found: {SELECTION_PATH}. Run this notebook "
            "with TARGET_SPLIT='validation' first."
        )
    trial_names = [json.loads(SELECTION_PATH.read_text())["selected_trial"]]

top_k_by_trial = {"generator_order": generator_top_k}
rows = [evaluate_pipeline(generator_top_k, "generator_order")]
for name in trial_names:
    top_k_by_trial[name] = rank_top_k(scored_candidates, TRIALS[name])
    rows.append(evaluate_pipeline(top_k_by_trial[name], name))
    print("Evaluated:", name)

results = pl.concat(rows, how="diagonal_relaxed")

if TARGET_SPLIT == "validation":
    trial_results = results.filter(pl.col("model").is_in(trial_names)).sort(
        SELECTION_METRIC, descending=True
    )
    selected_trial = trial_results["model"][0]
    SELECTION_PATH.write_text(
        json.dumps(
            {
                "split": "validation",
                "k": K,
                "selection_metric": SELECTION_METRIC,
                "selected_trial": selected_trial,
                "trial_scores": {
                    row["model"]: row[SELECTION_METRIC]
                    for row in trial_results.iter_rows(named=True)
                },
                "eval_impressions": int(n_selected),
                "seed": SEED,
                "candidate_path": str(CANDIDATE_PATH),
            },
            indent=2,
        )
    )
    print("Validation-selected content rule:", selected_trial)
else:
    selected_trial = trial_names[0]
    print("Using validation-selected content rule:", selected_trial)

final_top_k = top_k_by_trial[selected_trial]
final_top_k_path = (
    OUTPUT_DIR / f"{BASE_MODEL_NAME}_{TARGET_SPLIT}_top{K}_{IMPRESSION_TAG}.parquet"
)
final_top_k.write_parquet(final_top_k_path)
print("Final Top-K saved to:", final_top_k_path)

ordered = results.select(
    "model",
    "split",
    "k",
    "impressions",
    "clicked_impressions",
    f"ndcg_at_{K}",
    "mrr",
    *CG_METRICS,
).sort(f"ndcg_at_{K}", descending=True)
results_path = (
    RESULTS
    / f"pipeline_{CANDIDATE_ARTIFACTS.name}_content_similarity_"
    f"{TARGET_SPLIT}_{IMPRESSION_TAG}.csv"
)
ordered.write_csv(results_path)
print("Metrics saved to:", results_path)
display(ordered)

Evaluated: content_text_cosine
Evaluated: content_entity_cosine
Evaluated: content_fusion_text_0.25
Evaluated: content_fusion_text_0.50
Evaluated: content_fusion_text_0.75
Validation-selected content rule: content_entity_cosine
Final Top-K saved to: C:\Users\Yong Rui\BT4222-Group-6-\5_Models\5_03_Pipeline_Evaluation\artifacts\combined_content_two_tower_rolling_popularity_content_similarity_validation_top10_10000imp.parquet
Metrics saved to: C:\Users\Yong Rui\BT4222-Group-6-\results\pipeline_combined_content_two_tower_rolling_popularity_content_similarity_validation_10000imp.csv


model,split,k,impressions,clicked_impressions,ndcg_at_10,mrr,recall_pooled,recall_impression_mean,hit_rate,long_tail_recall,cold_article_recall,novelty_bits,catalogue_coverage,personalisation
str,str,i64,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""generator_order""","""validation""",10,10000,10000,0.222394,0.199035,0.313491,0.356991,0.4238,0.0,0.0,9.370233,0.001893,0.749435
"""content_entity_cosine""","""validation""",10,10000,10000,0.007008,0.00583,0.013339,0.014188,0.02,0.001983,0.0,15.007278,0.107682,0.982903
"""content_fusion_text_0.75""","""validation""",10,10000,10000,0.004613,0.003538,0.00949,0.010238,0.0142,0.001322,0.0,15.351249,0.151289,0.986256
"""content_text_cosine""","""validation""",10,10000,10000,0.00448,0.003577,0.009357,0.009898,0.0139,0.000661,0.0,15.297436,0.151247,0.989225
"""content_fusion_text_0.50""","""validation""",10,10000,10000,0.004234,0.003288,0.008494,0.009109,0.0127,0.001653,0.0,15.344231,0.141508,0.983519
"""content_fusion_text_0.25""","""validation""",10,10000,10000,0.004109,0.003045,0.008494,0.009064,0.0127,0.001653,0.0,15.233661,0.127161,0.982295


# 5. Notes and Limitations

- The popularity half is non-personalised: every request in the same hour gets the same
  base order, apart from availability. The content half is already personalised, and the
  content reranker reorders the combined 500 candidates.
- All metrics are known-click metrics. A generated article that was never shown has unknown
  relevance and can never count as a click, so the final list is partly rewarded for matching
  what the logging policy showed.
- `cg_evaluation.py` metrics at `M = K` describe the final recommendation list, not the
  500-candidate pool. `catalogue_coverage` and `personalisation` therefore measure the final
  Top-K, not retrieval depth.
- Validation chooses the content rule by nDCG@K. Test must reuse that saved selection; do not
  re-select after seeing test results.
- The test run needs `recommendations_test_top500.parquet` from
  `CG_content_two_tower_rolling_popularity.ipynb`. Run that notebook with
  `TARGET_SPLIT = "test"` once before changing this notebook to test.